# Normalizzazione personaggi DC e Marvel

In questa sezione procediamo all'unificazione e normalizzazione dei dataset dei personaggi provenienti da **DC Comics** (`PersonaggiDCFive_clean.csv`) e **Marvel Comics** (`PersonaggiMarvelFive_clean.csv`).

L'obiettivo è ottenere una struttura dati omogenea, priva di ambiguità e pronta per l'importazione nei nodi `:Character` del database a grafi **Neo4j**.


In [13]:
from pathlib import Path
import pandas as pd

# Definizione dei percorsi
BASE_PATH = Path('/Users/marcogreco/ComicGraph')
PULITI_PATH = BASE_PATH / 'Dataset' / 'Puliti'

# Caricamento dei dataset puliti dei personaggi
dc_characters = pd.read_csv(PULITI_PATH / 'PersonaggiDCFive_clean.csv')
marvel_characters = pd.read_csv(PULITI_PATH / 'PersonaggiMarvelFive_clean.csv')

# Stampa di verifica
print(f"DC Characters caricati: {len(dc_characters)} righe, {len(dc_characters.columns)} colonne")
print(f"Marvel Characters caricati: {len(marvel_characters)} righe, {len(marvel_characters.columns)} colonne")
print("\nColonne DC:", dc_characters.columns.tolist())
print("Colonne Marvel:", marvel_characters.columns.tolist())


DC Characters caricati: 6896 righe, 11 colonne
Marvel Characters caricati: 16373 righe, 11 colonne

Colonne DC: ['page_id', 'name', 'urlslug', 'ID', 'ALIGN', 'EYE', 'HAIR', 'SEX', 'ALIVE', 'APPEARANCES', 'Year First Appearance']
Colonne Marvel: ['page_id', 'name', 'urlslug', 'ID', 'ALIGN', 'EYE', 'HAIR', 'SEX', 'ALIVE', 'APPEARANCES', 'Year First Appearance']


In [14]:
#Aggiunta di una colonna relativa al publisher (per fare in modo che vengano divisi nei grafi)
dc_characters['publisher'] = 'DC'
marvel_characters['publisher'] = 'Marvel'

#Dobbiamo creare una chiave primaria univoca globale (id personaggio) cambiando il modo in cui sono indicizzati nel dataset originale
#manteniamo il valore assoluto dell'id originale ma aggiungiamo il valore "dc_<id_originale>" e "marvel_<id_originale>" in modo da evitare fusioni dello stesso id quando andremo a fondere i due dataset
dc_characters['character_id'] = 'dc_' + dc_characters['page_id'].astype(str)
marvel_characters['character_id'] = 'marvel_' + marvel_characters['page_id'].astype(str)

#verifichiamo che non ci siano sovrapposti mantendendo l'unicità dopo la modifica agli id
dc_ids = set(dc_characters['character_id'])
marvel_ids = set(marvel_characters['character_id'])
intersection = dc_ids.intersection(marvel_ids)
print(f"Totale ID DC creati: {len(dc_ids)}")
print(f"Totale ID Marvel creati: {len(marvel_ids)}")
print(f"Sovrapposizioni tra ID DC e Marvel: {len(intersection)}")






Totale ID DC creati: 6896
Totale ID Marvel creati: 16373
Sovrapposizioni tra ID DC e Marvel: 0


In [15]:
#Vado a modificare il nome delle colonne dei due dataset in modo che siano tutti scritti con lo "snake_case" per normalizzare il tutto
#riordino poi le colonne

# Dizionario di rinominazione delle colonne
column_mapping = {
    'page_id': 'original_page_id',
    'name': 'name',
    'urlslug': 'url_slug',
    'ID': 'identity_status',
    'ALIGN': 'alignment',
    'EYE': 'eye_color',
    'HAIR': 'hair_color',
    'SEX': 'gender',
    'ALIVE': 'status',
    'APPEARANCES': 'appearances_count',
    'Year First Appearance': 'first_appearance_year'
}

# Rinominazione colonne per entrambi i dataset
dc_characters = dc_characters.rename(columns=column_mapping)
marvel_characters = marvel_characters.rename(columns=column_mapping)

# Riordino delle colonne in una sequenza logica
desired_order = [
    'character_id', 
    'publisher', 
    'original_page_id', 
    'name', 
    'url_slug', 
    'identity_status', 
    'alignment', 
    'eye_color', 
    'hair_color', 
    'gender', 
    'status', 
    'appearances_count', 
    'first_appearance_year'
]

dc_characters = dc_characters[desired_order]
marvel_characters = marvel_characters[desired_order]

# Stampa di verifica
print("Colonne DC rinominate:", dc_characters.columns.tolist())
print("Colonne Marvel rinominate:", marvel_characters.columns.tolist())
print("\nVerifica uguaglianza colonne:", list(dc_characters.columns) == list(marvel_characters.columns))


Colonne DC rinominate: ['character_id', 'publisher', 'original_page_id', 'name', 'url_slug', 'identity_status', 'alignment', 'eye_color', 'hair_color', 'gender', 'status', 'appearances_count', 'first_appearance_year']
Colonne Marvel rinominate: ['character_id', 'publisher', 'original_page_id', 'name', 'url_slug', 'identity_status', 'alignment', 'eye_color', 'hair_color', 'gender', 'status', 'appearances_count', 'first_appearance_year']

Verifica uguaglianza colonne: True


In [16]:
# Rimozione delle parole ridondanti che qualificano inutilmente gli attributi
def clean_redundant_words(df):
    df_clean = df.copy()
    
    # 1. Rimozione di " Characters" nelle colonne alignment, gender e status
    for col in ['alignment', 'gender', 'status']:
        df_clean[col] = df_clean[col].str.replace(r'\s+Characters$', '', regex=True).str.strip()
    
    # 2. Rimozione di " Identity" sia alla fine (es. "Secret Identity" -> "Secret")
    #    sia all'inizio (es. "Identity Unknown" -> "Unknown")
    df_clean['identity_status'] = (
        df_clean['identity_status']
        .str.replace(r'\s+Identity$', '', regex=True)
        .str.replace(r'^Identity\s+', '', regex=True)
        .str.strip()
    )
    
    # 3. Rimozione di " Eyes", " Eyeballs" e " Eye" (singolare)
    #    (?<!\bNo) esclude il match se preceduto da "No" (preserva 'No Eyes')
    df_clean['eye_color'] = (
        df_clean['eye_color']
        .str.replace(r'(?<!\bNo)\s+(Eyes?|Eyeballs)$', '', regex=True)
        .str.strip()
    )
    
    # 4. Rimozione di " Hair" a fine stringa
    #    (?<!\bNo) esclude il match se preceduto da "No" (preserva 'No Hair')
    df_clean['hair_color'] = (
        df_clean['hair_color']
        .str.replace(r'(?<!\bNo)\s+Hair$', '', regex=True)
        .str.strip()
    )
    
    return df_clean


dc_characters = clean_redundant_words(dc_characters)
marvel_characters = clean_redundant_words(marvel_characters)



In [17]:
def check_redundant_words(df, dataset_name="Dataset"):
    print(f"=== CONTROLLO PAROLE RIDONDANTI: {dataset_name} ===")
    
    # 1. Controllo di 'Characters' in alignment, gender e status
    for col in ['alignment', 'gender', 'status']:
        found = df[df[col].astype(str).str.contains(r'\bcharacters\b', case=False, na=False)][col].unique()
        print(f"• '{col}' contiene 'Characters': {len(found)} valori {list(found)}")
        
    # 2. Controllo di 'Identity' in identity_status
    found_id = df[df['identity_status'].astype(str).str.contains(r'\bidentity\b', case=False, na=False)]['identity_status'].unique()
    print(f"• 'identity_status' contiene 'Identity': {len(found_id)} valori {list(found_id)}")
    
    # 3. Controllo di Eyes, Eyeballs, Eye e 'No' singolo in eye_color
    # Ignora 'No Eyes' ma trova qualsiasi altro residuo di Eye/Eyes/Eyeballs
    eyes_residue = df[df['eye_color'].astype(str).str.contains(r'(?<!\bNo\s)\b(?:eyes?|eyeballs)\b', case=False, na=False)]['eye_color'].unique()
    eyes_only_no = df[df['eye_color'].astype(str).str.strip().str.lower() == 'no']['eye_color'].unique()
    print(f"• 'eye_color' residui 'Eyes/Eye/Eyeballs': {len(eyes_residue)} valori {list(eyes_residue)}")
    print(f"• 'eye_color' stringhe isolate 'No': {len(eyes_only_no)} valori {list(eyes_only_no)}")
    
    # 4. Controllo di Hair e 'No' singolo in hair_color
    # Ignora 'No Hair' ma trova qualsiasi altro residuo di Hair
    hair_residue = df[df['hair_color'].astype(str).str.contains(r'(?<!\bNo\s)\bhair\b', case=False, na=False)]['hair_color'].unique()
    hair_only_no = df[df['hair_color'].astype(str).str.strip().str.lower() == 'no']['hair_color'].unique()
    print(f"• 'hair_color' residui 'Hair': {len(hair_residue)} valori {list(hair_residue)}")
    print(f"• 'hair_color' stringhe isolate 'No': {len(hair_only_no)} valori {list(hair_only_no)}")
    print()

# Esecuzione del controllo su DC e Marvel
check_redundant_words(dc_characters, "DC Comics")
check_redundant_words(marvel_characters, "Marvel Comics")


=== CONTROLLO PAROLE RIDONDANTI: DC Comics ===
• 'alignment' contiene 'Characters': 0 valori []
• 'gender' contiene 'Characters': 0 valori []
• 'status' contiene 'Characters': 0 valori []
• 'identity_status' contiene 'Identity': 0 valori []
• 'eye_color' residui 'Eyes/Eye/Eyeballs': 0 valori []
• 'eye_color' stringhe isolate 'No': 0 valori []
• 'hair_color' residui 'Hair': 0 valori []
• 'hair_color' stringhe isolate 'No': 0 valori []

=== CONTROLLO PAROLE RIDONDANTI: Marvel Comics ===
• 'alignment' contiene 'Characters': 0 valori []
• 'gender' contiene 'Characters': 0 valori []
• 'status' contiene 'Characters': 0 valori []
• 'identity_status' contiene 'Identity': 0 valori []
• 'eye_color' residui 'Eyes/Eye/Eyeballs': 0 valori []
• 'eye_color' stringhe isolate 'No': 0 valori []
• 'hair_color' residui 'Hair': 0 valori []
• 'hair_color' stringhe isolate 'No': 0 valori []



In [18]:
#Unifichiamo alcuni valori tra il dataset Marvel e DC





#Unificazione 'No Hair' -> 'Bald' per Marvel, DC presenta già i valori bald 
marvel_characters['hair_color'] = marvel_characters['hair_color'].replace('No Hair', 'Bald')

#unifichiamo "agender" nella colonna Gender di marvel in "genderless" di DC per la stessa colonna
marvel_characters['gender'] = marvel_characters['gender'].replace('Agender', 'Genderless')


#Colonne categoriche da uniformare con 'Unknown'
cat_cols = ['identity_status', 'alignment', 'eye_color', 'hair_color', 'gender', 'status']
#Sostituiamo tutti i NaN con la stringa 'Unknown' in entrambi i dataset
dc_characters[cat_cols] = dc_characters[cat_cols].fillna('Unknown')
marvel_characters[cat_cols] = marvel_characters[cat_cols].fillna('Unknown')

#unificazione identity_status marvel al dataset DC 
marvel_characters['identity_status'] = marvel_characters['identity_status'].replace({'No Dual' : 'Public', 'Known to Authorities': 'Secret'})

#unificazione alignment, 3 personaggi DC sono ex criminali, essendo una statistica piccola li unifichiamo come "good"
dc_characters['alignment'] = dc_characters['alignment'].replace({'Reformed Criminals': 'Good'})
 

#pulizia dello url_slug che è stato importato con corruzioni date dall'export json (probabilmente)
dc_characters['url_slug'] = dc_characters['url_slug'].str.replace(r'\/wiki\/', '', regex=False).str.replace(r'\/', '', regex=False)
marvel_characters['url_slug'] = marvel_characters['url_slug'].str.replace(r'\/', '', regex=False)
#aggiungiamo poi il dominio allo slug per ottenere un link cliccabile, i domini sono presi dalla documentazione del dataset dei personaggi e vengono dal sito wiki ufficiali di Fandom
dc_characters['wiki_url'] = 'https://dc.fandom.com/wiki/' + dc_characters['url_slug']
marvel_characters['wiki_url'] = 'https://marvel.fandom.com/wiki/' + marvel_characters['url_slug']


#pulizia delle corruzzioni date dall'export json
def clean_character_name(val):
    if not isinstance(val, str):
        return val
    # Decodifica dei caratteri unicode grezzi \u00e9 -> é, \u00f1 -> ñ ecc
    if '\\u' in val or '\\U' in val:
        try:
            val = val.encode('latin1').decode('unicode_escape')
        except Exception:
            pass
    # Rimozione dei backslash di escape davanti a virgolette \" -> " e barre \/ -> /
    val = val.replace('\\"', '"').replace('\\/', '/')
    return val.strip()


dc_characters['name'] = dc_characters['name'].apply(clean_character_name)
marvel_characters['name'] = marvel_characters['name'].apply(clean_character_name)





# Verifica valori capelli
print("Valori capelli Marvel:", sorted(marvel_characters['hair_color'].dropna().unique()))
# Verifica gender
print("Gender DC:", sorted(dc_characters['gender'].dropna().unique()))
print("Gender Marvel:", sorted(marvel_characters['gender'].dropna().unique()))
#controllo valori null
nan_dc = dc_characters[cat_cols].isna().sum().sum()
nan_marvel = marvel_characters[cat_cols].isna().sum().sum()
print(f"3. NaN residui nelle colonne categoriche -> DC: {nan_dc}, Marvel: {nan_marvel} (atteso: 0)")
#controllo identity status
print("4. Identity Status DC:", sorted(dc_characters['identity_status'].unique()))
print("   Identity Status Marvel:", sorted(marvel_characters['identity_status'].unique()))
#controllo alignment
reformed_dc = (dc_characters['alignment'] == 'Reformed Criminals').sum()
print(f"5. Residui 'Reformed Criminals' in DC: {reformed_dc} (atteso: 0)")
print("   • Alignment DC:", sorted(dc_characters['alignment'].unique()))
print("   • Alignment Marvel:", sorted(marvel_characters['alignment'].unique()))
#controllo url e link wiki
backslash_url_dc = dc_characters['url_slug'].str.contains(r'\\', regex=False).sum()
backslash_url_marvel = marvel_characters['url_slug'].str.contains(r'\\', regex=False).sum()
print(f"6. Backslash residui in url_slug -> DC: {backslash_url_dc}, Marvel: {backslash_url_marvel} (atteso: 0)")
print("   • Esempio wiki_url DC:", dc_characters['wiki_url'].iloc[0])
print("   • Esempio wiki_url Marvel:", marvel_characters['wiki_url'].iloc[0])
#controllo nomi
backslash_name_dc = dc_characters['name'].str.contains(r'\\', regex=False).sum()
backslash_name_marvel = marvel_characters['name'].str.contains(r'\\', regex=False).sum()
print(f"7. Backslash residui nei nomi -> DC: {backslash_name_dc}, Marvel: {backslash_name_marvel} (atteso: 0)")



Valori capelli Marvel: ['Auburn', 'Bald', 'Black', 'Blond', 'Blue', 'Bronze', 'Brown', 'Dyed', 'Gold', 'Green', 'Grey', 'Light Brown', 'Magenta', 'Orange', 'Orange-brown', 'Pink', 'Purple', 'Red', 'Reddish Blond', 'Silver', 'Strawberry Blond', 'Unknown', 'Variable', 'White', 'Yellow']
Gender DC: ['Female', 'Genderless', 'Male', 'Transgender', 'Unknown']
Gender Marvel: ['Female', 'Genderfluid', 'Genderless', 'Male', 'Unknown']
3. NaN residui nelle colonne categoriche -> DC: 0, Marvel: 0 (atteso: 0)
4. Identity Status DC: ['Public', 'Secret', 'Unknown']
   Identity Status Marvel: ['Public', 'Secret', 'Unknown']
5. Residui 'Reformed Criminals' in DC: 0 (atteso: 0)
   • Alignment DC: ['Bad', 'Good', 'Neutral', 'Unknown']
   • Alignment Marvel: ['Bad', 'Good', 'Neutral', 'Unknown']
6. Backslash residui in url_slug -> DC: 0, Marvel: 0 (atteso: 0)
   • Esempio wiki_url DC: https://dc.fandom.com/wiki/Batman_(Bruce_Wayne)
   • Esempio wiki_url Marvel: https://marvel.fandom.com/wiki/Spider-Man_(

In [19]:
# Conversione delle colonne numeriche a interi, sono colonne inerenti all'anno di prima uscita e il numero di comparizioni che sono discreti non possono essere float
numeric_cols = ['appearances_count', 'first_appearance_year']

for col in numeric_cols:
    dc_characters[col] = dc_characters[col].astype('Int64')
    marvel_characters[col] = marvel_characters[col].astype('Int64')


print("Tipi di dato DC:\n", dc_characters[numeric_cols].dtypes)
print("\nTipi di dato Marvel:\n", marvel_characters[numeric_cols].dtypes)
print("\nAnteprima DC:\n", dc_characters[['name', 'appearances_count', 'first_appearance_year']].head())


Tipi di dato DC:
 appearances_count        Int64
first_appearance_year    Int64
dtype: object

Tipi di dato Marvel:
 appearances_count        Int64
first_appearance_year    Int64
dtype: object

Anteprima DC:
                           name  appearances_count  first_appearance_year
0         Batman (Bruce Wayne)               3093                   1939
1        Superman (Clark Kent)               2496                   1986
2   Green Lantern (Hal Jordan)               1565                   1959
3     James Gordon (New Earth)               1316                   1987
4  Richard Grayson (New Earth)               1237                   1940


In [21]:
# 1. Unione dei due DataFrame (DC + Marvel)
characters_final = pd.concat([dc_characters, marvel_characters], ignore_index=True)
# 2. Definizione del percorso di output
OUTPUT_PATH = PULITI_PATH / 'Personaggi_DC_Marvel_Norm.csv'
# 3. Esportazione in formato CSV
characters_final.to_csv(OUTPUT_PATH, index=False)
# 4. Stampe di conferma
print("=" * 60)
print(f"Dataset finale unificato salvato con successo in:\n-> {OUTPUT_PATH}")
print(f"• Dimensioni totali: {characters_final.shape[0]} righe, {characters_final.shape[1]} colonne")
print("• Distribuzione publisher:")
print(characters_final['publisher'].value_counts().to_string())
print("=" * 60)
print("\nAnteprima prime 3 righe (DC):")
print(characters_final.head(3)[['character_id', 'publisher', 'name', 'alignment', 'identity_status', 'appearances_count', 'first_appearance_year']])
print("\nAnteprima ultime 3 righe (Marvel):")
print(characters_final.tail(3)[['character_id', 'publisher', 'name', 'alignment', 'identity_status', 'appearances_count', 'first_appearance_year']])

Dataset finale unificato salvato con successo in:
-> /Users/marcogreco/ComicGraph/Dataset/Puliti/Personaggi_DC_Marvel_Norm.csv
• Dimensioni totali: 23269 righe, 14 colonne
• Distribuzione publisher:
publisher
Marvel    16373
DC         6896

Anteprima prime 3 righe (DC):
  character_id publisher                        name alignment  \
0      dc_1422        DC        Batman (Bruce Wayne)      Good   
1     dc_23387        DC       Superman (Clark Kent)      Good   
2      dc_1458        DC  Green Lantern (Hal Jordan)      Good   

  identity_status  appearances_count  first_appearance_year  
0          Secret               3093                   1939  
1          Secret               2496                   1986  
2          Secret               1565                   1959  

Anteprima ultime 3 righe (Marvel):
        character_id publisher                            name alignment  \
23266  marvel_695217    Marvel   Tinkerer (Skrull) (Earth-616)       Bad   
23267  marvel_708811    Mar